# Minimal RAG (Retrieval-Augmented Generation) Solution

**RAG** enhances Large Language Models by retrieving relevant documents from a knowledge base before generating responses. This grounds the LLM's answers in actual sources instead of relying solely on training data.

## Components

| Component | Technology | Description |
|-----------|------------|-------------|
| Data Source | Wikipedia | Public knowledge base for demo |
| Text Splitter | LangChain | Breaks documents into chunks |
| Embeddings | `all-MiniLM-L6-v2` | Converts text to vectors |
| Vector Store | FAISS | Fast similarity search |
| LLM | Ollama (Llama 3.2) | Local language model |

## Prerequisites 
#### (Windows OS)

**1. Install dependencies:**
```bash
pip install -r requirements.txt
```

**2. Install Ollama** from https://ollama.com/download, then:
```bash
ollama pull llama3.2      # 3B model (~2GB)
# OR
ollama pull llama3.2:1b   # 1B model (~1.3GB, faster)
(Try & "$env:LOCALAPPDATA\Programs\Ollama\ollama.exe" pull llama3.2:1b if not working)
```

---

## Step 0: Setup & Imports

In [1]:
import os
import warnings

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import WikipediaLoader

from src.splitter import split_documents
from src.vectorstore import get_vectorstore_retriever
from src.rag import get_rag_chain

print("All imports successful")

All imports successful


---

## Step 1: Data Ingestion

Load raw data from various sources into the RAG system. Here we fetch Wikipedia articles about AI topics.

In [2]:
topics = [
    "World war 2",
    "United Nations",
    "World war 1"
]

# Limit fetched data to avoid large downloads
MAX_DOCS_PER_TOPIC = 1       # Number of documents per topic
MAX_CHARS_PER_DOC = 200_000     # Max characters per document

docs = []
for topic in topics:
    print(f"Loading: {topic}...")
    loader = WikipediaLoader(
        query=topic,
        load_max_docs=MAX_DOCS_PER_TOPIC,
        doc_content_chars_max=MAX_CHARS_PER_DOC
    )
    docs.extend(loader.load())

print(f"\nLoaded {len(docs)} documents")
print(f"Total characters: {sum(len(d.page_content) for d in docs):,}")
print(f"Sample source: {docs[0].metadata.get('source', 'N/A')}")

Loading: World war 2...
Loading: United Nations...
Loading: World war 1...

Loaded 3 documents
Total characters: 236,770
Sample source: https://en.wikipedia.org/wiki/World_War_II


---

## Step 2: Text Splitting (Chunking)

Break documents into smaller chunks that fit embedding model limits and enable precise retrieval.

```
Original Document (10,000 chars)
         |
         v
[Chunk 1] [Chunk 2] [Chunk 3] ...
 1000 ch   1000 ch   1000 ch
     |--overlap--|
```

Overlap ensures context isn't lost at split boundaries.

In [3]:
splits = split_documents(docs=docs)

print(f"Split {len(docs)} documents into {len(splits)} chunks")
print(f"Average chunk size: {sum(len(s.page_content) for s in splits) // len(splits)} characters")

Split 3 documents into 180 chunks
Average chunk size: 1322 characters


---

## Step 3: Embeddings & Vector Store

**Embeddings** convert text into numerical vectors that capture semantic meaning. Similar texts produce similar vectors.

**Vector Store** (FAISS) indexes these embeddings for fast similarity search.

```
"The cat sat on the mat"   -> [0.12, -0.45, 0.78, ...] (384 dims)
"A feline rested on a rug" -> [0.11, -0.44, 0.79, ...] (similar!)
```

In [4]:
print("Creating embeddings and vector store...")

retriever = get_vectorstore_retriever(
    documents=splits,
    persist_directory="./faiss_index",
    k=3
)

print("Vector store created and saved to ./faiss_index")

Creating embeddings and vector store...


c:\Users\ADola\Desktop\minimal-rag\src\vectorstore.py:29: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name, model_kwargs={'device': 'cpu'})
W0129 00:58:54.116000 25372 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Vector store created and saved to ./faiss_index


---

## Step 4: RAG Chain Setup

The RAG chain connects: Query -> Retriever -> Prompt Template -> LLM -> Response

| Model | Parameters | Speed |
|-------|------------|-------|
| `llama3.2:1b` | 1B | Fast |
| `llama3.2` | 3B | Medium |

In [5]:
MODEL_NAME = "llama3.2:1b"  # "llama3.2:1b" for faster inference

print(f"Setting up RAG chain with model: {MODEL_NAME}")

rag_chain = get_rag_chain(
    retriever=retriever,
    model_name=MODEL_NAME
)

print("RAG chain ready")

Setting up RAG chain with model: llama3.2:1b
RAG chain ready


---

## Step 5: Query the RAG System

Now we can ask questions and get responses grounded in our retrieved documents.

In [ ]:
query = "Years gap between World War 1 and World War 2?"

print(f"Query: {query}")
print("\nProcessing...\n")

response = rag_chain.invoke(query)

print("Response:")
print("-" * 50)
print(response)

Query: What is Retrieval-augmented generation and how does it work?

Processing...



ResponseError: model 'llama3.2:1b' not found (status code: 404)

In [ ]:
# View retrieved sources
print("Sources Retrieved:")
print("=" * 50)

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, 1):
    title = doc.metadata.get('title', 'Unknown')
    source = doc.metadata.get('source', 'N/A')
    print(f"\n{i}. {title}")
    print(f"   Source: {source}")
    print(f"   Preview: {doc.page_content[:150]}...")

---

## Try Your Own Questions

In [8]:
my_query = "Explain when WW1 has started?"

print(f"Query: {my_query}\n")
response = rag_chain.invoke(my_query)
print(f"Response:\n{response}")

Query: Explain when WW1 has started?

Response:
I don't have enough information to answer the question. The context provided does not mention when World War I started, only that it began on July 28, 1914, with the assassination of Archduke Franz Ferdinand by Gavrilo Princip in Sarajevo, Bosnia.


---

## Summary

This notebook demonstrated a complete RAG pipeline:

1. **Data Ingestion** - Loaded Wikipedia articles
2. **Text Splitting** - Chunked documents (1000 chars, 200 overlap)
3. **Embeddings** - Converted to vectors using MiniLM
4. **Vector Store** - Indexed in FAISS for retrieval
5. **RAG Chain** - Connected retriever to LLM for grounded responses